In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC

from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

In [2]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

df.isnull().sum()

df["Churn"].value_counts()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [4]:
# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Drop missing values
df = df.dropna()

In [5]:
customer_ids = df["customerID"]

df = df.drop("customerID", axis=1)

In [6]:
df["Churn"] = df["Churn"].map({"No":0, "Yes":1})

In [7]:
df = pd.get_dummies(df, drop_first=True)

In [8]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

In [9]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [11]:
C_values = [0.1, 1, 10]

results = []

In [12]:
for C in C_values:
    
    svm = SVC(kernel='linear', C=C, probability=True)
    
    svm.fit(X_train, y_train)
    
    predictions = svm.predict(X_test)
    
    probs = svm.predict_proba(X_test)[:,1]
    
    acc = accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions)
    rec = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    roc = roc_auc_score(y_test, probs)
    
    cm = confusion_matrix(y_test, predictions)
    
    support_vectors = svm.support_vectors_.shape[0]
    
    print("\nC =", C)
    print("Confusion Matrix:\n", cm)
    print("Accuracy:", acc)
    print("Precision:", prec)
    print("Recall:", rec)
    print("F1 Score:", f1)
    print("ROC AUC:", roc)
    print("Support Vectors:", support_vectors)
    
    results.append([C, acc, prec, rec, f1, roc, support_vectors])


C = 0.1
Confusion Matrix:
 [[916 117]
 [169 205]]
Accuracy: 0.7967306325515281
Precision: 0.6366459627329193
Recall: 0.5481283422459893
F1 Score: 0.5890804597701149
ROC AUC: 0.8274650439248127
Support Vectors: 2576

C = 1
Confusion Matrix:
 [[917 116]
 [167 207]]
Accuracy: 0.798862828713575
Precision: 0.6408668730650154
Recall: 0.553475935828877
F1 Score: 0.593974175035868
ROC AUC: 0.8272450316041229
Support Vectors: 2570

C = 10
Confusion Matrix:
 [[917 116]
 [168 206]]
Accuracy: 0.7981520966595593
Precision: 0.639751552795031
Recall: 0.5508021390374331
F1 Score: 0.5919540229885057
ROC AUC: 0.8269797226291731
Support Vectors: 2569


In [13]:
results_df = pd.DataFrame(
    results,
    columns=[
        "C",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC_AUC",
        "Support_Vectors"
    ]
)

results_df

,C,Accuracy,Precision,Recall,F1,ROC_AUC,Support_Vectors
0,0.1,0.796731,0.636646,0.548128,0.589080,0.827465,2576
1,1.0,0.798863,0.640867,0.553476,0.593974,0.827245,2570
2,10.0,0.798152,0.639752,0.550802,0.591954,0.826980,2569


In [14]:
results_df.to_csv("svm_linear_results.csv", index=False)

In [15]:
best_model = SVC(kernel="linear", C=1, probability=True)

best_model.fit(X_train, y_train)

preds = best_model.predict(X_test)

probs = best_model.predict_proba(X_test)[:,1]

pred_df = pd.DataFrame({
    "Actual": y_test,
    "Predicted": preds,
    "Score": probs
})

pred_df.to_csv("test_predictions.csv", index=False)

pred_df.head()

,Actual,Predicted,Score
974,0,0,0.030917
619,0,1,0.657967
4289,0,0,0.050424
3721,1,0,0.195687
4533,0,0,0.127413
